### Esse notebook tem como objetivo utilizar as principais funcoes do spark usando arquivos delta mas apenas spark sql

In [1]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F

# Pacotes necessários (incluindo Delta)
packages = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.2",
    "com.amazonaws:aws-java-sdk-bundle:1.12.628"
])

builder = SparkSession.builder \
    .appName("DeltaLakeApp") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.jars.packages", packages) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print(f"✅ Spark {spark.version} com Delta Lake configurado!")

✅ Spark 3.5.0 com Delta Lake configurado!


In [2]:
spark

In [3]:
nam_path_s3 = 's3a://datalake/tmp_data/netflix_titles_sql/'

In [5]:
def execute_sql(command,num_lines=10):
    spark.sql(command).show(num_lines)

### Criando tabela em delta

In [6]:
df_pyspark=spark.read.csv('s3a://datalake/raw_data/netflix_titles.csv', inferSchema=True, header=True, sep=',', quote='"', escape='"', multiLine=True)

In [7]:
df_pyspark.createOrReplaceTempView('netflix_titles_sql')

In [8]:
execute_sql('show tables')

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  default|    netflix_titles|      false|
|         |netflix_titles_sql|      false|
+---------+------------------+-----------+



In [9]:
execute_sql('create schema if not exists gold')
execute_sql('show schemas')

++
||
++
++

+---------+
|namespace|
+---------+
|  default|
|     gold|
+---------+



In [9]:
execute_sql(f'''
    CREATE OR REPLACE TABLE gold.netflix_titles_sql
        USING DELTA
        LOCATION '{nam_path_s3}' AS
            SELECT * FROM netflix_titles_sql''');

++
||
++
++



In [10]:
### Validacao de contagem de registro
execute_sql('select count (*) from gold.netflix_titles_sql', 15)
df_pyspark.agg(F.count('*')).show(10)

+--------+
|count(1)|
+--------+
|    8807|
+--------+

+--------+
|count(1)|
+--------+
|    8807|
+--------+



In [11]:
execute_sql('describe table gold.netflix_titles_sql',20)

+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|     show_id|   string|   NULL|
|        type|   string|   NULL|
|       title|   string|   NULL|
|    director|   string|   NULL|
|        cast|   string|   NULL|
|     country|   string|   NULL|
|  date_added|   string|   NULL|
|release_year|      int|   NULL|
|      rating|   string|   NULL|
|    duration|   string|   NULL|
|   listed_in|   string|   NULL|
| description|   string|   NULL|
+------------+---------+-------+



### Leitura de dados

In [12]:
execute_sql('''
    select type, 
           title 
           from gold.netflix_titles_sql 
                order by 1 asc''',10)

+-----+--------------------+
| type|               title|
+-----+--------------------+
|Movie|        Je Suis Karl|
|Movie|            Paranoia|
|Movie|Confessions of an...|
|Movie|     Avvai Shanmughi|
|Movie|          Dark Skies|
|Movie|Go! Go! Cory Cars...|
|Movie|             Sankofa|
|Movie|               Jeans|
|Movie|Dick Johnson Is Dead|
|Movie|      Minsara Kanavu|
+-----+--------------------+
only showing top 10 rows



### Update dos dados

In [13]:
execute_sql('select type, count (*) from gold.netflix_titles_sql group by 1')

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  Movie|    6131|
+-------+--------+



In [14]:
execute_sql("""
    update gold.netflix_titles_sql
        set type = 'movie'
        where type ='Movie'
""")

+-----------------+
|num_affected_rows|
+-----------------+
|             6131|
+-----------------+



In [15]:
execute_sql('select type, count (*) from gold.netflix_titles_sql group by 1')

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  movie|    6131|
+-------+--------+



### Update e insert de dados (Merge)

In [16]:
df_pyspark_update=spark.read.csv('s3a://datalake/raw_data/netflix_titles_augmented.csv', inferSchema=True, header=True, sep=',', quote='"', escape='"', multiLine=True)
df_pyspark_update.createOrReplaceTempView('netflix_titles_sql_update')

In [17]:
execute_sql('select type, count (*) from gold.netflix_titles_sql group by 1')

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  movie|    6131|
+-------+--------+



In [18]:
execute_sql('''
                MERGE INTO gold.netflix_titles_sql AS t1
                USING netflix_titles_sql_update as t2
                ON t1.show_id = t2.show_id
                WHEN MATCHED
                  THEN UPDATE SET *
                WHEN NOT MATCHED
                  THEN INSERT *
            ''')

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             9807|            8807|               0|             1000|
+-----------------+----------------+----------------+-----------------+



In [19]:
execute_sql('select type, count (*) from gold.netflix_titles_sql group by 1')

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    3154|
|  Movie|    6653|
+-------+--------+



In [20]:
execute_sql('select count (*) from gold.netflix_titles_sql')

+--------+
|count(1)|
+--------+
|    9807|
+--------+



### Delete dos Dados

In [21]:
execute_sql('delete from gold.netflix_titles_sql where show_id=\'s1\' ')

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [22]:
execute_sql('select count (*) from gold.netflix_titles_sql where show_id=\'s1\'')

+--------+
|count(1)|
+--------+
|       0|
+--------+



### Rename de coluna

In [23]:
execute_sql("""
    ALTER TABLE gold.netflix_titles_sql SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")

++
||
++
++



In [24]:
execute_sql("""
    ALTER TABLE gold.netflix_titles_sql RENAME COLUMN director TO diretor
""")


++
||
++
++



In [25]:
execute_sql('select director from gold.netflix_titles_sql')

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `director` cannot be resolved. Did you mean one of the following? [`diretor`, `cast`, `duration`, `country`, `rating`].; line 1 pos 7;
'Project ['director]
+- SubqueryAlias spark_catalog.gold.netflix_titles_sql
   +- Relation spark_catalog.gold.netflix_titles_sql[show_id#7856,type#7857,title#7858,diretor#7859,cast#7860,country#7861,date_added#7862,release_year#7863,rating#7864,duration#7865,listed_in#7866,description#7867] parquet


In [26]:
execute_sql('select diretor from gold.netflix_titles_sql')

+----------------+
|         diretor|
+----------------+
|            NULL|
|            NULL|
|            NULL|
|     Abhinay Deo|
|       Kiran Rao|
|Satyajit Bhatkal|
|      Aamir Khan|
|            NULL|
| Taranveer Singh|
|            NULL|
+----------------+
only showing top 10 rows



### Adicao de coluna

In [27]:
execute_sql("""
    ALTER TABLE gold.netflix_titles_sql add COLUMNs (dat_load TIMESTAMP) 
""")

++
||
++
++



In [28]:
execute_sql("""
    update gold.netflix_titles_sql
        set dat_load = current_timestamp()
""")

+-----------------+
|num_affected_rows|
+-----------------+
|             9806|
+-----------------+



In [29]:
execute_sql('describe gold.netflix_titles_sql',20)

+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|     show_id|   string|   NULL|
|        type|   string|   NULL|
|       title|   string|   NULL|
|     diretor|   string|   NULL|
|        cast|   string|   NULL|
|     country|   string|   NULL|
|  date_added|   string|   NULL|
|release_year|      int|   NULL|
|      rating|   string|   NULL|
|    duration|   string|   NULL|
|   listed_in|   string|   NULL|
| description|   string|   NULL|
|    dat_load|timestamp|   NULL|
+------------+---------+-------+



In [30]:
execute_sql(""" select current_timestamp()
""")

+--------------------+
| current_timestamp()|
+--------------------+
|2026-01-16 14:39:...|
+--------------------+



### Drop de coluna

In [31]:
spark.sql("""
    ALTER TABLE gold.netflix_titles_sql DROP COLUMN diretor
""")

print("✅ Coluna deletada com sucesso!")

# Verifique o resultado
spark.sql("DESCRIBE TABLE gold.netflix_titles_sql").show()

✅ Coluna deletada com sucesso!
+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|     show_id|   string|   NULL|
|        type|   string|   NULL|
|       title|   string|   NULL|
|        cast|   string|   NULL|
|     country|   string|   NULL|
|  date_added|   string|   NULL|
|release_year|      int|   NULL|
|      rating|   string|   NULL|
|    duration|   string|   NULL|
|   listed_in|   string|   NULL|
| description|   string|   NULL|
|    dat_load|timestamp|   NULL|
+------------+---------+-------+



### Recuperação de Histórico

In [15]:
execute_sql('DESCRIBE HISTORY gold.netflix_titles_sql;',100)

+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|     21|2026-01-16 14:39:06|  NULL|    NULL|        DROP COLUMNS|{columns -> ["dir...|NULL|    NULL|     NULL|         20|  Serializable|         true|                  {}|        NULL|Apache-Spark/3.5....|
|     20|2026-01-16 14:38:56|  NULL|    NULL|              UPDATE|   {predicate -> []}|NULL|    NULL|     NULL|         19|  Serializable|        false|{numRemovedFiles

In [16]:
execute_sql('SELECT * FROM gold.netflix_titles_sql VERSION AS OF 2;')

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan